# 02 1024D 임베딩과 품질 입력

명시적으로 등록한 완료 export로 CUDA 추출합니다. CR-FIQA를 기본 품질 신호로 사용하고 raw norm도 보존합니다. 학습은 FaceX-Zoo 03에서 수행합니다.


In [1]:
# 1. 사용자 설정
# 목적: 임베딩 추출 대상·저장 위치·실행 여부와 선택적 DB 저장을 설정합니다.

from pathlib import Path
import sys, os
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research").is_dir())  # 현재 ronbun 위치를 자동 탐색하는 내부 경로입니다. 직접 바꾸지 않습니다.
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
CONFIG_PATH = PROJECT_ROOT / "configs/experiments/facex_workflow.yaml"  # 실험 YAML 경로입니다. 다른 실험 설정을 쓰려면 같은 workflow 형식의 YAML을 지정합니다.
# FACE_DATASET_ROOT/FACE_MODEL_ROOT 환경변수가 우선합니다. 미지정 시 Windows F:/, Linux /data/ 기본값을 읽습니다.
from research.runtime.storage import roots
STORAGE = roots()  # 환경변수/OS 기본값에서 읽은 경로입니다. 실제 경로 변경은 아래 두 ROOT 변수에서 합니다.
DATASET_ROOT = Path(STORAGE["DATASET_ROOT"])  # 공유 데이터 루트; 필요하면 Path("실제 데이터 경로")로 변경합니다. Windows 기본 F:/face_datasets.
MODEL_ROOT = Path(STORAGE["MODEL_ROOT"])  # 공유 모델 루트; 필요하면 Path("실제 모델 경로")로 변경합니다. Windows 기본 F:/models.
os.environ["FACE_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FACE_MODEL_ROOT"] = str(MODEL_ROOT)
DATASETS = None  # CONFIG_PATH의 datasets 사용. 부분 실행 시 데이터셋 목록을 지정합니다.
EXECUTE = True  # False=상태/계획 확인, True=아래 셀에서 실제 작업 실행입니다.
OUTPUT_ROOT = DATASET_ROOT.parent / "ronbun_results/facex_workflow"  # 입력 cache와 실험 결과의 공통 저장 루트입니다. 02·03에서 같은 경로를 사용합니다.
PERSIST_ORIGIN_TO_DB = False  # False=DB 저장 안 함, True=추출 후 Origin 저장; EXECUTE=True와 images/pgvector 준비가 필요합니다.

# FaceX 04가 출력한 완료 model inventory를 사용하려면 정확한 경로를 지정합니다.
# MODEL_INVENTORY = None  # 예: Path(r"F:/models/facex/selected/ucface_recipes_v1/inventories/<hash>.json")
MODEL_INVENTORY = Path(
    "F:/models/facex/selected/ucface_recipes_v1/inventories/"
    "a0d381e891e1daec6db2314abfb1a0ed85fbdd92c5118026d895deea965c27e6.json"
)
INVENTORY_CONFIG_PATH = OUTPUT_ROOT / "configs/selected_models_v2_local.yaml"  # 목록 변경 시 새 파일명. 03의 CONFIG_PATH에도 이 경로 사용.
INVENTORY_PURPOSE = "local_protocol_baseline"  # v2 일반 loss 모델. 진단은 diagnostic_control, 과거 v1은 legacy_local_control로 명시.
INVENTORY_ORIGIN_ONLY = True  # 새 모델은 압축 전 Origin부터 비교. 이후 False + 새 YAML 경로로 PCA/PQ 실험.
MODEL_REGISTRY_ROOT = MODEL_ROOT / "facex/ronbun_registry"


In [2]:
# 2. 임베딩·품질 입력 준비
# 목적: 계획을 확인하거나 완료 export로 1024D 임베딩·품질 입력을 만들고 필요 시 DB에 저장합니다.

from research.experiments.facex_workflow import inspect_workflow, prepare_inputs
if MODEL_INVENTORY is not None and EXECUTE:
    from research.experiments.facex_inventory import write_inventory_config
    CONFIG_PATH = write_inventory_config(MODEL_INVENTORY, CONFIG_PATH, MODEL_REGISTRY_ROOT, INVENTORY_CONFIG_PATH, purpose=INVENTORY_PURPOSE, origin_only=INVENTORY_ORIGIN_ONLY)
    print("03에서 사용할 CONFIG_PATH:", CONFIG_PATH)
if EXECUTE:
    input_record = prepare_inputs(CONFIG_PATH, datasets=DATASETS, output_root=OUTPUT_ROOT)
    display(input_record)
    from research.experiments.facex_comparison import verify_alignment_ranks
    if verified := verify_alignment_ranks(input_record["input_index"]):
        print("정렬 대조 Rank-1 일치 (정답 수, 질의 수)", verified)
    if PERSIST_ORIGIN_TO_DB:
        import json
        from research.database.facex import persist_origin_input
        records = json.loads(Path(input_record["input_index"]).read_text(encoding="utf-8"))
        for entry in records["inputs"].values():
            display(persist_origin_input(entry["path"]))
elif MODEL_INVENTORY is not None:
    print("계획 모드: inventory 등록/입력 추출 미실행", MODEL_INVENTORY)
    print("생성 예정 config:", INVENTORY_CONFIG_PATH)
else:
    display(inspect_workflow(CONFIG_PATH, datasets=DATASETS))


03에서 사용할 CONFIG_PATH: F:\ronbun_results\facex_workflow\configs\selected_models_v2_local.yaml


{'input_index': 'F:\\ronbun_results\\facex_workflow\\input_indexes\\152cdcca00dd461c0b99f323bd985ed7626a1a58fb01da342e2681a36573d27a.json',
 'input_count': 12,
 'estimated_input_bytes': 10421270784}

정렬 대조 Rank-1 일치 (정답 수, 질의 수) {'scface_arcface_two_eye_alignment_s8972': {False: (1020, 1200)}, 'tinyface_arcface_pfld_five_point_alignment_s8972': {False: (2297, 3728), True: (2071, 3728)}}
